In [ ]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

In [ ]:
# Connect to client libraries

openai = OpenAI()


openrouter_url = "https://openrouter.ai/api/v1"
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

In [ ]:
models = ["gpt-5", "x-ai/grok-4", "google/gemini-2.5-pro", "openai/gpt-oss-120b", ]

clients = {"gpt-5": openai, "x-ai/grok-4": openrouter, "google/gemini-2.5-pro": openrouter, "openai/gpt-oss-120b": openrouter}

In [ ]:
LANG_CONFIG = {
    "py": {
        "name": "Python",
        "extension": "py",
    },
    "cpp": {
        "name": "C++",
        "extension": "cpp",
        "compile_command": ["cl", "main.cpp", "/O2", "/Fe:main.exe"],
        "run_command": [".\\main.exe"],
    },
    "rs": {
        "name": "Rust",
        "extension": "rs",
        "compile_command": [
            "rustc", "main.rs",
            "-C", "opt-level=3",
            "-C", "target-cpu=native",
            "-o", "main.exe",
        ],
        "run_command": [".\\main.exe"],
    },
}

In [ ]:
def system_prompt_for(lang_from, lang_to):
    from_name = LANG_CONFIG[lang_from]["name"]
    to_name = LANG_CONFIG[lang_to]["name"]

    return f"""
Convert {from_name} code into high performance {to_name} code.

Respond ONLY with valid {to_name} code.
Do NOT include markdown, explanations, or text outside the code.

Requirements:
- The output must produce identical results.
- Optimize for performance (use O(n) where possible instead of O(n^2)).
- The code must compile and run without errors.

Language-specific rules:

For Rust:
- Use proper Rust formatting syntax (e.g. {{:.6}} instead of {{:.6f}})
- Use `println!` correctly
- Use `fn main()` as entry point

For C++:
- Use standard C++ (MSVC compatible)
- Include necessary headers (iostream, vector, etc.)
- Use `int main()`

General:
- Do NOT use Python-specific syntax
- Do NOT assume dynamic typing
- Ensure all variables are properly declared

Return ONLY the code.
"""

In [ ]:
def user_prompt_for(code, lang_from, lang_to):
    from_name = LANG_CONFIG[lang_from]["name"]
    to_name = LANG_CONFIG[lang_to]["name"]
    ext = LANG_CONFIG[lang_to]["extension"]

    compile_cmd = LANG_CONFIG[lang_to].get("compile_command")

    if compile_cmd:
        compile_text = (
            f"\nThe output will be written to main.{ext} and compiled using:\n"
            f"{compile_cmd}\n"
        )
    else:
        compile_text = (
            f"\nThe output will be written to main.{ext} and executed directly.\n"
        )

    return (
        f"Convert the following {from_name} code into {to_name}.\n\n"
        f"{compile_text}\n"
        f"Respond ONLY with {to_name} code.\n\n"
        f"Input code:\n\n"
        f"```{from_name.lower()}\n"
        f"{code}\n"
        f"```"
    )

In [ ]:
def messages_for(code, lang_from, lang_to):
    return [
        {"role": "system", "content": system_prompt_for(lang_from, lang_to)},
        {"role": "user", "content": user_prompt_for(code, lang_from, lang_to)}
    ]

In [ ]:
def write_output(code, lang_key):
    ext = LANG_CONFIG[lang_key]["extension"]
    with open(f"main.{ext}", "w", encoding="utf-8") as f:
        f.write(code)

In [ ]:
def clean_code(code):
    # Extract code from markdown block if present
    if "```" in code:
        parts = code.split("```")
        if len(parts) >= 2:
            code = parts[1]

            # remove language tag (cpp, rust, python, etc.)
            lines = code.split("\n")
            if lines and lines[0].strip().isalpha():
                code = "\n".join(lines[1:])

    return code.strip()

In [ ]:
def port(model, code, lang_from, lang_to):
    client = clients[model]

    reasoning_effort = "high" if "gpt" in model else None

    response = client.chat.completions.create(
        model=model,
        messages=messages_for(code, lang_from, lang_to),
        reasoning_effort=reasoning_effort
    )

    reply = response.choices[0].message.content

    reply = clean_code(reply)

    return reply

In [ ]:
def run_python(code):
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, {"__builtins__": __builtins__})
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output or "<empty>"

In [ ]:
def run_code(code, lang_key):
    if lang_key == "py":
        return run_python(code)
    else:
        return compile_and_run(code, lang_key)

In [ ]:
def compile_and_run(code, lang_to):
    cfg = LANG_CONFIG[lang_to]

    try:
        
        write_output(code, lang_to)

        
        if lang_to == "py":
            return run_python(code)

        
        compile_result = subprocess.run(
            cfg["compile_command"],
            check=True,
            text=True,
            capture_output=True
        )
        print("Compile STDERR:", compile_result.stderr)

        
        run_result = subprocess.run(
            cfg["run_command"],
            check=True,
            text=True,
            capture_output=True
        )
        print("Run STDERR:", run_result.stderr)

        return run_result.stdout or "<empty>"

    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [ ]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [ ]:
from styles import CSS
import gradio as gr

LANG_OPTIONS = [
    ("Python", "py"),
    ("C++", "cpp"),
    ("Rust", "rs")
]

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Programming Language Converter") as ui:

    # 🎯 Title
    gr.Markdown(
    """
    <h1 style='text-align: center; margin-bottom: 10px;'>
        Programming Language Converter
    </h1>
    <p style='text-align: center; color: gray;'>
        Convert, optimize and run code across Python, C++, and Rust
    </p>
    """
)

    # 🔽 Language selectors
    with gr.Row():
        lang_from = gr.Dropdown(
            choices=LANG_OPTIONS,
            value="py",
            label="From Language"
        )
        lang_to = gr.Dropdown(
            choices=LANG_OPTIONS,
            value="rs",
            label="To Language"
        )

    # 🧠 Code panels
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            input_code = gr.Code(
                label="Input Code",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            output_code = gr.Code(
                label="Generated Code",
                value="",
                language="cpp",  # ✅ must be "rs", not "rust"
                lines=26
            )

    # 🎛️ Controls
    with gr.Row(elem_classes=["controls"]):
        run_input = gr.Button("Run Input", elem_classes=["run-btn"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button("Convert", elem_classes=["convert-btn"])
        run_output = gr.Button("Run Output", elem_classes=["run-btn"])

    # 📊 Outputs
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            input_out = gr.TextArea(label="Input Result", lines=8)
        with gr.Column(scale=6):
            output_out = gr.TextArea(label="Output Result", lines=8)

    # 🔧 Wrappers (connect UI → backend)

    def convert_wrapper(model, code, lang_from_val, lang_to_val):
        return port(model, code, lang_from_val, lang_to_val)

    def run_input_wrapper(code, lang_from_val):
        return run_code(code, lang_from_val)

    def run_output_wrapper(code, lang_to_val):
        return compile_and_run(code, lang_to_val)

    # 🔗 Bind buttons

    convert.click(
        fn=convert_wrapper,
        inputs=[model, input_code, lang_from, lang_to],
        outputs=[output_code]
    )

    run_input.click(
        fn=run_input_wrapper,
        inputs=[input_code, lang_from],
        outputs=[input_out]
    )

    run_output.click(
        fn=run_output_wrapper,
        inputs=[output_code, lang_to],
        outputs=[output_out]
    )

ui.launch(inbrowser=True, share=True)